In [16]:
# =====================================================================
# 1. Import Libraries
# =====================================================================
import pandas as pd
import numpy as np
import os

# =====================================================================
# REFERENCE_YEAR: ثابت واحد للسنة "الحالية" المستخدمة في فحوصات
# invalid-year / outlier عبر Part 2 و Part 3 (بدل datetime.now().year)،
# عشان النتائج تفضل reproducible حتى لو حد شغّل الكود لاحقًا في سنة تانية.
# =====================================================================
REFERENCE_YEAR = 2026

# =====================================================================
# 2. Define Dataset Files  (filename -> Make)
# =====================================================================
# عدّل DATA_DIR حسب مكان الملفات الفعلي عندك في Colab
DATA_DIR = ""   # مثلاً "/content/" أو مسار Drive لو mounted

FILE_TO_MAKE = {
    "audi.csv":     "Audi",
    "bmw.csv":      "BMW",
    "ford.csv":     "Ford",
    "hyundi.csv":   "Hyundai",
    "merc.csv":     "Mercedes",
    "skoda.csv":    "Skoda",
    "toyota.csv":   "Toyota",
    "vauxhall.csv": "Vauxhall",
    "vw.csv":       "Volkswagen",
}

# =====================================================================
# 3. Load Datasets  +  4. Add Make Column
#    (بيحصلوا مع بعض، لكل ملف لوحده، قبل الدمج)
# =====================================================================
raw_dataframes = {}     # كل ملف خام لوحده، من غير أي تعديل على قيمه
columns_by_make = {}    # عشان نقارن أسماء الأعمدة قبل الدمج

for filename, make in FILE_TO_MAKE.items():
    path = os.path.join(DATA_DIR, filename)

    if not os.path.exists(path):
        print(f"⚠️  File not found, skipped: {path}")
        continue

    df = pd.read_csv(path)
    columns_by_make[make] = set(df.columns)   # سجّل الأعمدة قبل إضافة Make
    df["Make"] = make
    raw_dataframes[make] = df

# ---- فحص اتساق أسماء الأعمدة بين الملفات قبل الدمج ----
all_columns    = set.union(*columns_by_make.values())
common_columns = set.intersection(*columns_by_make.values())

print("Columns common to ALL files:", sorted(common_columns))
print("Columns NOT present in every file:", sorted(all_columns - common_columns))

for make, cols in columns_by_make.items():
    missing_here = all_columns - cols
    if missing_here:
        print(f"  {make} does not have: {sorted(missing_here)}  (actual columns: {sorted(cols)})")

# لو الفحص فوق ظهرله اختلاف اسم (الداتاست ده معروف إنه بيحصل فيه أحيانًا)،
# صحّح اسم العمود فقط هنا -- قبل الدمج -- حسب اللي هتشوفه فعليًا مطبوع فوق.
# مثال (فكّ التعليق وعدّله حسب اللي تشوفه):
# raw_dataframes["Hyundai"].rename(columns={"tax(£)": "tax"}, inplace=True)
# ---- Standardize known column-name differences BEFORE combining ----
# مؤكد من نتيجة التشغيل: كل الملفات بتستخدم "tax" ماعدا Hyundai اللي بتستخدم "tax(£)"
COLUMN_RENAME_MAP = {
    "tax(£)": "tax",
}

for make, df in raw_dataframes.items():
    for old_name, new_name in COLUMN_RENAME_MAP.items():
        if old_name in df.columns and new_name not in df.columns:
            df.rename(columns={old_name: new_name}, inplace=True)
            print(f"  Renamed '{old_name}' -> '{new_name}' in {make}")
# =====================================================================
# 5. Combine Datasets
# =====================================================================
combined_cars = pd.concat(raw_dataframes.values(), ignore_index=True)

# =====================================================================
# حفظ الملف المدموج
# =====================================================================
# الملفات الأصلية فوق ملهاش أي تعديل. to_csv() بتعمل overwrite كاملة
# مش append، يعني كل مرة تشغّل النوتبوك من الأول، الملف بيتكتب من جديد.
OUTPUT_PATH = "combined_cars.csv"
combined_cars.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {combined_cars.shape[0]} rows x {combined_cars.shape[1]} columns to {OUTPUT_PATH}")

# =====================================================================
# 6. Dataset Size
# =====================================================================
print("Total rows:", combined_cars.shape[0])
print("Total columns:", combined_cars.shape[1])
print("\nRows per manufacturer:")
print(combined_cars["Make"].value_counts())

# =====================================================================
# 7. Columns
# =====================================================================
print(combined_cars.columns.tolist())

# =====================================================================
# 8. Data Types
# =====================================================================
print(combined_cars.dtypes)

# =====================================================================
# 9. Missing Values
# =====================================================================
missing_count = combined_cars.isnull().sum()
missing_pct   = (missing_count / len(combined_cars) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing_count, "missing_pct(%)": missing_pct})
print(missing_report[missing_report["missing_count"] > 0])

# =====================================================================
# 10. Duplicates
# =====================================================================
print("Duplicate rows in combined_cars:", combined_cars.duplicated().sum())

print("\nDuplicate rows inside each original file (before combining):")
for make, df in raw_dataframes.items():
    dupes = df.drop(columns=["Make"]).duplicated().sum()
    if dupes:
        print(f"  {make}: {dupes} duplicate rows")

# =====================================================================
# 11. Numerical Statistics
# =====================================================================
print(combined_cars.describe())

# دالة بسيطة تلاقي العمود بغض النظر عن حالة الأحرف/المسافات
def find_col(df, *name_options):
    normalized = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}
    for name in name_options:
        key = name.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

price_col   = find_col(combined_cars, "price")
mileage_col = find_col(combined_cars, "mileage")
year_col    = find_col(combined_cars, "year")

for label, col in [("price", price_col), ("mileage", mileage_col), ("year", year_col)]:
    if col:
        print(f"{label} ({col}): min={combined_cars[col].min()}, max={combined_cars[col].max()}, "
              f"zeros={(combined_cars[col] == 0).sum()}, negatives={(combined_cars[col] < 0).sum()}")

# =====================================================================
# 12. Categorical Values
# =====================================================================
categorical_cols = combined_cars.select_dtypes(include="object").columns.tolist()
print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    if combined_cars[col].nunique() > 30:   # زي "model" اللي قيمه كتير
        print(f"\n{col}: {combined_cars[col].nunique()} unique values")
        print(combined_cars[col].value_counts().head(20))
    else:
        print(f"\n{col} unique values:", sorted(combined_cars[col].dropna().unique().tolist()))

Columns common to ALL files: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'transmission', 'year']
Columns NOT present in every file: ['tax', 'tax(£)']
  Audi does not have: ['tax(£)']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'tax', 'transmission', 'year'])
  BMW does not have: ['tax(£)']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'tax', 'transmission', 'year'])
  Ford does not have: ['tax(£)']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'tax', 'transmission', 'year'])
  Hyundai does not have: ['tax']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'tax(£)', 'transmission', 'year'])
  Mercedes does not have: ['tax(£)']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'model', 'mpg', 'price', 'tax', 'transmission', 'year'])
  Skoda does not have: ['tax(£)']  (actual columns: ['engineSize', 'fuelType', 'mileage', 'mode